In [ ]:
import boto3, shutil, os, subprocess, time

region = boto3.Session().region_name or "us-east-1"
bucket = "my-agent-artifacts-bucket"          # an existing bucket in the same region
key = f"hello-agent/code-{int(time.time())}.zip"   # new key per build forces a redeploy

shutil.rmtree("build", ignore_errors=True)
os.makedirs("build")
# AgentCore runs on ARM64 (Graviton), so pull aarch64 wheels
subprocess.run([
    "pip", "install", "-q", "-r", "requirements.txt", "--target", "build",
    "--platform", "manylinux2014_aarch64", "--only-binary=:all:",
    "--python-version", "3.12",
], check=True)
shutil.copy("hello_agent.py", "build/")
shutil.make_archive("hello_agent", "zip", "build")

boto3.client("s3").upload_file("hello_agent.zip", bucket, key)
print(f"s3://{bucket}/{key}")